# 投资研究问题生成系统 v2 — 逐步教程

## 相比 v1 的改进

| | v1 | v2 |
|---|---|---|
| **迭代轮次** | 3 轮 | 6 轮 |
| **迭代模板** | 第 2-3 轮只在"补充" | 每轮完整循环：思考→提问→回应→更新 |
| **分析师角色** | 三人角度相似，问题大量重复 | 允许交叉但要求独特视角切入 |
| **最终报告** | 仅问题清单 | 三部分：问题清单 + 共识与分歧 + 执行路线图 |

## 系统做什么？

输入一家公司名称（如"拼多多"），3 个不同视角的分析师 Agent 会：
1. **第 1 轮**：各自独立提出 8-10 个关键研究问题
2. **第 2-6 轮**：每轮完成完整的思考循环——
   - 【思考】反思其他分析师的视角
   - 【提问】就问题清单本身向其他分析师提出疑问
   - 【回应】回应其他分析师对自己的疑问
   - 【更新】补充盲区、删除重复、重排优先级
3. **最后**：总结 Agent 生成可执行的研究计划

```
3 个分析师 × 6 轮 + 1 个总结 = 19 次 LLM 调用
```

---

下面我们一步步构建这个系统，每个 Cell 都可以单独运行。

## 第 1 步：安装依赖

如果你还没安装过这些库，取消注释下面的 `!pip install` 运行一次即可。

In [ ]:
%%capture --no-stderr
# 安装所有需要的依赖包
# langgraph: 图工作流框架
# langchain_openai: OpenAI 集成
# langchain_google_genai: Google Gemini 集成
# langchain_core: LangChain 核心
%pip install --quiet -U langchain-google-genai langgraph langchain_openai langchain_core

## 第 2 步：配置 API Key

你需要一个 Google 或 OpenAI 的 API Key。两种方式任选一种：
- **方式 A**：在同目录下创建 `.env` 文件，写入 `GOOGLE_API_KEY=xxx...` 或 `OPENAI_API_KEY=sk-xxx...`
- **方式 B**：直接在下面的代码里填入（不推荐提交到 Git）

In [ ]:
import os
from dotenv import load_dotenv

# 从 .env 文件加载环境变量（如果有的话）
load_dotenv()

# 如果 .env 里没有配置，可以在这里直接填入（记得不要提交到 Git）
# os.environ["GOOGLE_API_KEY"] = "xxx..."
# os.environ["OPENAI_API_KEY"] = "sk-xxx..."

# 检查 API Key 是否已配置
google_key = os.environ.get("GOOGLE_API_KEY", "")
openai_key = os.environ.get("OPENAI_API_KEY", "")

if google_key:
    print(f"✅ Google API Key 已配置 (前 8 位: {google_key[:8]}...)")
elif openai_key:
    print(f"✅ OpenAI API Key 已配置 (前 8 位: {openai_key[:8]}...)")
else:
    print("❌ 未检测到 API Key，请先配置 GOOGLE_API_KEY 或 OPENAI_API_KEY")

## 第 3 步：定义 State（状态）

LangGraph 的核心概念是 **State** —— 所有数据都存在这个共享状态里，每个节点读取和更新它。

我们的 State 包含：
| 字段 | 含义 |
|------|------|
| `company` | 要研究的公司名 |
| `round_number` | 当前第几轮（1→2→...→6） |
| `value_analyst_history` | 价值分析师每轮的输出列表 |
| `growth_analyst_history` | 成长分析师每轮的输出列表 |
| `risk_analyst_history` | 风险分析师每轮的输出列表 |
| `final_questions` | 最终结构化问题清单 |

In [ ]:
from typing import TypedDict


class ResearchState(TypedDict):
    company: str                        # 要研究的公司名称
    round_number: int                   # 当前轮次 (1, 2, ..., 6)
    value_analyst_history: list[str]    # 价值分析师每轮的输出
    growth_analyst_history: list[str]   # 成长分析师每轮的输出
    risk_analyst_history: list[str]     # 风险分析师每轮的输出
    final_questions: str                # 总结Agent的最终输出


print("✅ State 定义完成")
print("字段:", list(ResearchState.__annotations__.keys()))

## 第 4 步：编写 Prompt 模板

### 角色设计思路（v2 改进）

v1 的问题：三个分析师都在讨论同样的话题（如 Temu、竞争、管理层），角度相似，导致大量重复。

v2 的解决方案：**允许讨论同一话题，但要求从各自独特的分析框架切入**。例如同样讨论 Temu：
- 价值分析师：Temu 的护城河在哪？资本回报率能否转正？
- 成长分析师：Temu 的 TAM 有多大？用户留存和复购数据如何？
- 风险分析师：Temu 面临的关税封禁风险有多大？

### 模板设计思路（v2 改进）

v1 只有"补充"没有"辩论"。v2 每轮都是完整的思考循环：
- 【思考】反思其他分析师的视角
- 【提问】就问题清单向其他分析师提出疑问（指名提问）
- 【回应】回应其他分析师的疑问
- 【更新】更新自己的问题列表

随着轮次推进，早期自然偏补充（盲区多），后期自然偏收敛（问题趋于稳定）。

In [ ]:
# ============================
# 角色系统提示 (System Prompt)
# ============================

VALUE_ANALYST_SYSTEM = """你是一位价值投资分析师，信奉巴菲特和芒格的投资哲学。

你的独特视角——所有问题都必须从以下角度切入：
- 商业模式的本质：公司到底靠什么赚钱，模式是否可持续
- 护城河质量：网络效应、转换成本、规模经济、品牌的深度和持久性
- 管理层诚信与资本配置：管理层是否值得信赖，钱花得是否明智
- 自由现金流与资本回报率：公司能产生多少真金白银
- 内在价值与安全边际：当前价格相对于内在价值是否合理

重要：你可以讨论任何话题（包括新业务、竞争、监管等），但必须始终从
"商业模式本质、护城河、现金流、安全边际"的视角提问，而不是泛泛而谈。
与其他分析师讨论同一话题时，你的问题角度应该明显不同。

你的任务是：针对给定的公司，列出你认为最关键的 8-10 个问题，按重要性排序。
只列问题，不要回答。每个问题附上一句话说明为什么这个问题重要。"""

GROWTH_ANALYST_SYSTEM = """你是一位成长型投资分析师，关注企业未来 3-5 年的增长潜力。

你的独特视角——所有问题都必须从以下角度切入：
- TAM 与渗透率：市场有多大，公司还能吃多少
- 增长驱动力与增长质量：营收增速靠什么驱动，是否可持续
- 用户/客户指标：活跃用户、ARPU、留存、复购的趋势
- 新业务与海外扩张：第二增长曲线的空间和进展
- 竞争格局中的份额变化：公司在行业中是赢是输

重要：你可以讨论任何话题（包括护城河、风险、管理层等），但必须始终从
"增长数据、市场空间、用户指标、竞争份额"的视角提问，而不是泛泛而谈。
与其他分析师讨论同一话题时，你的问题角度应该明显不同。

你的任务是：针对给定的公司，列出你认为最关键的 8-10 个问题，按重要性排序。
只列问题，不要回答。每个问题附上一句话说明为什么这个问题重要。"""

RISK_ANALYST_SYSTEM = """你是一位专注于风险评估的分析师，你的职责是找出所有可能导致投资失败的因素。

你的独特视角——所有问题都必须从以下角度切入：
- 监管与政策风险：反垄断、关税、数据安全、合规处罚
- 财务风险：会计操纵可能性、现金流质量、信息披露透明度
- 外部威胁：地缘政治、宏观经济冲击、技术替代
- 估值风险：市场预期是否过高、是否是价值陷阱
- 治理与关键人物风险：公司治理结构、核心人物依赖

重要：你可以讨论任何话题（包括商业模式、增长、新业务等），但必须始终从
"什么可能出错、风险有多大、最坏情况是什么"的视角提问，而不是泛泛而谈。
与其他分析师讨论同一话题时，你的问题角度应该明显不同。

你的任务是：针对给定的公司，列出你认为最关键的 8-10 个问题，按重要性排序。
只列问题，不要回答。每个问题附上一句话说明为什么这个问题重要。"""

SYNTHESIZER_SYSTEM = """你是一位高级研究主管。三位分析师经过多轮补充和辩论后，产出了最终的研究问题。

你的任务是生成一份可直接执行的研究计划，包含以下三个部分：

## 第一部分：核心研究问题清单（不超过 25 个问题）
- 去重、合并相似问题
- 按主题分类（如：商业模式、增长驱动、财务质量、竞争格局、风险因素、估值）
- 在每个分类内按重要性排序
- 每个问题标注：
  - 优先级：P0（必须搞清楚否则无法决策）/ P1（重要）/ P2（有帮助）
  - 数据来源：[财报] [行业研究] [管理层] [公开数据API]

## 第二部分：分析师共识与分歧
- 三位分析师都认为重要的问题有哪些（高置信度）
- 分析师之间有分歧的问题有哪些，分歧点是什么（需要特别关注）

## 第三部分：建议研究执行顺序
- 将 P0 问题按照"先查数据能回答的 → 再查行业研究的 → 最后需要主观判断的"排序
- 给出一个 3 步走的研究路线图：
  1. 第一步（定量）：哪些问题可以通过财报数据和公开 API 回答
  2. 第二步（定性）：哪些问题需要阅读行业报告、竞品分析
  3. 第三步（判断）：哪些问题需要综合以上信息做主观判断

不要回答这些问题，只做整理、排序和规划。"""

print("✅ 4 个角色的系统提示已定义")
print(f"  - 价值分析师: {len(VALUE_ANALYST_SYSTEM)} 字")
print(f"  - 成长分析师: {len(GROWTH_ANALYST_SYSTEM)} 字")
print(f"  - 风险分析师: {len(RISK_ANALYST_SYSTEM)} 字")
print(f"  - 总结主管:   {len(SYNTHESIZER_SYSTEM)} 字")

In [ ]:
# ============================
# 轮次提示模板
# ============================

# 第 1 轮：独立思考，只给公司名
ROUND_1_TEMPLATE = """公司: {company}

请列出你认为在决定是否投资这家公司之前必须搞清楚的关键问题。
列出你认为最关键的 8-10 个问题，按重要性排序。宁缺毋滥。"""

# 第 2-6 轮：统一迭代模板（思考→提问→回应→更新）
ROUND_N_TEMPLATE = """公司: {company}
当前是第 {round_number} 轮讨论（共 6 轮）。

你之前提出了以下问题：
{own_previous}

其他分析师的最新问题和讨论：
---
价值分析师: {value_prev}
成长分析师: {growth_prev}
风险分析师: {risk_prev}
---

请完成以下步骤：

【思考】看到其他分析师的视角后，你有什么新的发现或反思？

【提问】就问题清单本身，向其他分析师提出你的疑问
（指名提问，例如：
 "我想问成长分析师：你为什么没有把 XX 列入问题清单？这难道不是评估增长质量的关键吗？"
 "我想问价值分析师：你列的 XX 问题和 YY 问题是否本质上在问同一件事？"
 "我想问风险分析师：你把 XX 排在第一优先级，但我认为 YY 才是更根本的前置问题，你怎么看？"）

【回应】如果其他分析师在上一轮对你的问题清单提出了疑问，请回应

【更新】综合以上思考，更新你的问题列表：
- 补充你发现的盲区
- 删掉你认为不再重要或与他人重复的问题
- 重新排列优先级
- 输出你更新后的完整问题列表（控制在 8-10 个）"""

# 总结 Agent 的提示模板
SYNTHESIZER_TEMPLATE = """公司: {company}

以下是三位分析师经过六轮讨论后提出的所有研究问题和讨论：

{all_discussions}

请整理并输出一份可直接执行的研究计划。"""

print("✅ 提示模板已定义")
print("  - 第 1 轮模板（独立思考，8-10 个问题）")
print("  - 第 2-6 轮模板（思考→提问→回应→更新）")
print("  - 总结模板（三部分可执行报告）")

## 第 5 步：初始化 LLM

使用 LangChain 的 `ChatGoogleGenerativeAI` 封装来调用 Gemini。
你也可以切换为 `ChatOpenAI` 使用 OpenAI 的模型。

> **小知识**：`SystemMessage` 是告诉 LLM "你是谁"，`HumanMessage` 是用户的具体问题。

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# 使用 Gemini（默认）或 OpenAI，取消注释切换
llm = ChatGoogleGenerativeAI(model="gemini-3.1-pro-preview", temperature=0)
# llm = ChatOpenAI(model="gpt-4o")

# 获取模型名称（兼容不同 LLM 的属性名）
model_name = getattr(llm, "model", None) or getattr(llm, "model_name", "unknown")
print(f"✅ LLM 初始化完成: {model_name}")

## 第 6 步：实现各个节点（Node）

LangGraph 中，每个 **节点** 就是一个函数，输入是 State，输出是要更新的 State 字段。

我们需要这些节点：

| 节点 | 做什么 |
|------|--------|
| `value_analyst_node` | 价值分析师调用 LLM 生成问题 |
| `growth_analyst_node` | 成长分析师调用 LLM 生成问题 |
| `risk_analyst_node` | 风险分析师调用 LLM 生成问题 |
| `aggregate_node` | 轮次 +1（不调 LLM） |
| `synthesizer_node` | 总结所有讨论，生成最终研究计划 |

三个分析师的逻辑几乎一样，所以我们先写一个 **通用函数** `_analyst_node`，再用它创建三个分析师。

In [ ]:
def _get_response_text(response):
    """
    兼容不同 LLM 的响应格式。
    - Google Gemini: response.content 是 list[dict]，取 [0]['text']
    - OpenAI: response.content 是 str
    """
    content = response.content
    if isinstance(content, list):
        return content[0]["text"]
    return content


def _analyst_node(
    state: ResearchState,
    system_prompt: str,
    history_key: str,
) -> dict:
    """
    通用分析师节点。

    参数:
        state: 当前状态
        system_prompt: 这个分析师的角色提示（"你是谁"）
        history_key: 要更新的历史字段名（如 "value_analyst_history"）

    返回:
        更新后的历史字段
    """
    round_num = state["round_number"]
    company = state["company"]

    # 第 1 轮：独立思考
    if round_num == 1:
        user_msg = ROUND_1_TEMPLATE.format(company=company)
    # 第 2-6 轮：思考→提问→回应→更新
    else:
        user_msg = ROUND_N_TEMPLATE.format(
            company=company,
            round_number=round_num,
            own_previous=state[history_key][-1],
            value_prev=state["value_analyst_history"][-1],
            growth_prev=state["growth_analyst_history"][-1],
            risk_prev=state["risk_analyst_history"][-1],
        )

    # 调用 LLM
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_msg),
    ])

    # 把这一轮的输出追加到历史记录
    new_history = state[history_key] + [_get_response_text(response)]
    return {history_key: new_history}


print("✅ 通用分析师节点函数已定义")

In [ ]:
# 用通用函数创建三个具体的分析师节点

def value_analyst_node(state: ResearchState) -> dict:
    """价值分析师：关注护城河、现金流、安全边际"""
    print(f"  [价值分析师] 第 {state['round_number']} 轮分析中...")
    return _analyst_node(state, VALUE_ANALYST_SYSTEM, "value_analyst_history")


def growth_analyst_node(state: ResearchState) -> dict:
    """成长分析师：关注 TAM、增速、用户指标"""
    print(f"  [成长分析师] 第 {state['round_number']} 轮分析中...")
    return _analyst_node(state, GROWTH_ANALYST_SYSTEM, "growth_analyst_history")


def risk_analyst_node(state: ResearchState) -> dict:
    """风险分析师：关注监管、财务风险、外部威胁"""
    print(f"  [风险分析师] 第 {state['round_number']} 轮分析中...")
    return _analyst_node(state, RISK_ANALYST_SYSTEM, "risk_analyst_history")


print("✅ 三个分析师节点已定义: value / growth / risk")

In [ ]:
def aggregate_node(state: ResearchState) -> dict:
    """
    汇总节点：一轮结束后，轮次 +1。
    不调用 LLM，只更新 round_number。
    """
    new_round = state["round_number"] + 1
    print(f"  [汇总] 第 {state['round_number']} 轮完成 → 进入第 {new_round} 轮")
    return {"round_number": new_round}


def synthesizer_node(state: ResearchState) -> dict:
    """
    总结节点：综合六轮所有讨论内容，调用 LLM 生成可执行的研究计划。
    """
    print("  [总结Agent] 正在综合所有讨论，生成可执行的研究计划...")

    # 把六轮的所有输出拼接起来
    all_outputs = []
    for i in range(len(state["value_analyst_history"])):
        all_outputs.append(f"=== 第{i+1}轮 ===")
        all_outputs.append(f"价值分析师:\n{state['value_analyst_history'][i]}")
        all_outputs.append(f"成长分析师:\n{state['growth_analyst_history'][i]}")
        all_outputs.append(f"风险分析师:\n{state['risk_analyst_history'][i]}")

    user_msg = SYNTHESIZER_TEMPLATE.format(
        company=state["company"],
        all_discussions="\n\n".join(all_outputs),
    )

    response = llm.invoke([
        SystemMessage(content=SYNTHESIZER_SYSTEM),
        HumanMessage(content=user_msg),
    ])

    return {"final_questions": _get_response_text(response)}


print("✅ 汇总节点 + 总结节点已定义")

## 第 7 步：构建 LangGraph 图

这是最核心的部分！我们要把节点连接成一个图：

```
         ┌→ 价值分析师 ─┐
START ──→├→ 成长分析师 ─┤→ 汇总 ──→ 还没到6轮？──→ 回到三个分析师（循环）
         └→ 风险分析师 ─┘              │
                                    到6轮了？──→ 总结Agent ──→ END
```

**关键概念：**
- `add_edge(A, B)`：A 完成后走到 B
- `add_conditional_edges(A, func)`：A 完成后根据 func 的返回值决定走哪
- `Send("节点名", state)`：fan-out，把同一个 state 同时发送给多个节点并行执行

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

MAX_ROUNDS = 6  # 迭代 6 轮


def route_after_aggregate(state: ResearchState) -> list[Send] | str:
    """
    汇总后的路由逻辑：
    - 还没到 6 轮 → 用 Send 把 state 同时发给三个分析师（并行 fan-out）
    - 到了 6 轮 → 走向总结节点
    """
    if state["round_number"] > MAX_ROUNDS:
        print("  [路由] 六轮完成 → 进入总结阶段")
        return "synthesizer"

    print(f"  [路由] 继续第 {state['round_number']} 轮")
    return [
        Send("value_analyst", state),   # 并行发给价值分析师
        Send("growth_analyst", state),  # 并行发给成长分析师
        Send("risk_analyst", state),    # 并行发给风险分析师
    ]


print("✅ 路由函数已定义")

In [ ]:
# ============================
# 组装整个图
# ============================

graph_builder = StateGraph(ResearchState)

# 1. 添加所有节点
graph_builder.add_node("value_analyst", value_analyst_node)
graph_builder.add_node("growth_analyst", growth_analyst_node)
graph_builder.add_node("risk_analyst", risk_analyst_node)
graph_builder.add_node("aggregate", aggregate_node)
graph_builder.add_node("synthesizer", synthesizer_node)

# 2. 连接边：START → 三个分析师并行
graph_builder.add_edge(START, "value_analyst")
graph_builder.add_edge(START, "growth_analyst")
graph_builder.add_edge(START, "risk_analyst")

# 3. 连接边：三个分析师 → 汇总（fan-in，等三个都完成）
graph_builder.add_edge("value_analyst", "aggregate")
graph_builder.add_edge("growth_analyst", "aggregate")
graph_builder.add_edge("risk_analyst", "aggregate")

# 4. 条件边：汇总 → 路由（继续迭代 or 总结）
graph_builder.add_conditional_edges("aggregate", route_after_aggregate)

# 5. 连接边：总结 → END
graph_builder.add_edge("synthesizer", END)

# 6. 编译图
graph = graph_builder.compile()

print("✅ LangGraph 图构建完成！")
print("节点:", [n for n in graph.get_graph().nodes if not n.startswith("__")])

## 第 8 步（可选）：可视化图结构

运行下面的 Cell 可以看到图的 Mermaid 图表描述。如果你的 Notebook 环境支持，还可以直接渲染成图片。

In [ ]:
# 打印图的 Mermaid 格式描述
print(graph.get_graph().draw_mermaid())

# 如果你安装了 IPython，可以用下面的代码直接渲染：
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

## 第 9 步：运行！

现在万事俱备，设置要分析的公司名称，然后运行图。

> **预计耗时**：6 轮 × 3 个分析师 + 1 个总结 = 19 次 LLM 调用，大约 3-8 分钟。
>
> **预计成本**：使用 Gemini 3.1 Pro 大约 $1-2。

In [ ]:
# ============================
# 设置要分析的公司（改成你想研究的公司）
# ============================
COMPANY = ""

# 初始状态
initial_state = {
    "company": COMPANY,
    "round_number": 1,
    "value_analyst_history": [],
    "growth_analyst_history": [],
    "risk_analyst_history": [],
    "final_questions": "",
}

print(f"将要分析的公司: {COMPANY}")
print(f"初始状态: round_number={initial_state['round_number']}, 各历史记录为空")
print(f"迭代轮次: {MAX_ROUNDS} 轮")
print(f"\n开始运行...\n{'='*50}")

# 运行整个图！
result = graph.invoke(initial_state)

print(f"{'='*50}")
print("运行完成！")

## 第 10 步：查看最终结果

运行完成后，`result` 里就包含了所有数据。最终报告包含三部分：
1. **核心研究问题清单**（≤25 个，按主题分类，标注优先级和数据来源）
2. **分析师共识与分歧**（高置信度问题 + 需要特别关注的分歧）
3. **建议研究执行顺序**（定量→定性→判断的三步走路线图）

In [ ]:
# 查看最终结构化研究计划
from IPython.display import Markdown, display

display(Markdown(f"# {COMPANY} — 投资研究计划\n\n{result['final_questions']}"))

## 第 11 步（可选）：查看每轮讨论细节

如果你想看每个分析师在每一轮说了什么——包括他们的思考、互相提问和回应，运行下面的 Cell。

In [ ]:
# 查看每轮讨论的详细内容，并保存到文件
analysts = {
    "价值分析师": result["value_analyst_history"],
    "成长分析师": result["growth_analyst_history"],
    "风险分析师": result["risk_analyst_history"],
}

all_md = f"# {COMPANY} — 各轮讨论过程\n\n"

for round_idx in range(len(result["value_analyst_history"])):
    md = f"## 第 {round_idx + 1} 轮\n\n"
    for name, history in analysts.items():
        md += f"### {name}\n\n{history[round_idx]}\n\n---\n\n"
    all_md += md
    display(Markdown(md))

# 保存到文件
process_file = f"process_{COMPANY}.md"
with open(process_file, "w", encoding="utf-8") as f:
    f.write(all_md)
print(f"✅ 讨论过程已保存到: {process_file}")

## 第 12 步（可选）：保存结果到文件

In [ ]:
# 保存最终结果到 Markdown 文件
output_file = f"output_{COMPANY}.md"

with open(output_file, "w", encoding="utf-8") as f:
    f.write(f"# {COMPANY} — 投资研究计划\n\n")
    f.write(str(result["final_questions"]))

print(f"✅ 结果已保存到: {output_file}")